# Transport registry

Transport registries define public-transport mode parameters used to estimate edge travel time.


In [1]:
# To install IduEdu in a clean environment:
# !pip install IduEdu

OSM_ID = 1114252


In [2]:
from iduedu import DEFAULT_REGISTRY, DEFAULT_REGISTRY_W_TRAIN

print(DEFAULT_REGISTRY.list_types())
print(DEFAULT_REGISTRY_W_TRAIN.list_types())


['bus', 'trolleybus', 'tram', 'subway']
['bus', 'trolleybus', 'tram', 'subway', 'train']


## Inspect a transport specification

Each `TransportSpec` stores speed, acceleration, braking, and traffic parameters for one mode.


In [3]:
bus = DEFAULT_REGISTRY.get("bus")
print(bus)
print("Travel time for 800 m:", round(bus.travel_time_min(800), 2), "min")


TransportSpec(name='bus', vmax_tech_kmh=90, accel_dist_m=700, brake_dist_m=650, traffic_coef=0.7)
Travel time for 800 m: 2.57 min


## Create a custom registry

Use a custom registry when you need project-specific speed assumptions or additional transport modes.


In [4]:
from iduedu import TransportRegistry, TransportSpec

registry = TransportRegistry()
registry.add(
    TransportSpec(
        name="bus",
        vmax_tech_kmh=80,
        accel_dist_m=220,
        brake_dist_m=160,
        traffic_coef=0.7,
    )
)
registry.add(
    TransportSpec(
        name="tram",
        vmax_tech_kmh=70,
        accel_dist_m=180,
        brake_dist_m=140,
        traffic_coef=0.85,
    )
)

registry.list_types()


['bus', 'tram']

## Update, ensure, and remove specs

Names are normalized internally. `ensure` creates a missing mode from defaults when needed.


In [5]:
registry.update("bus", traffic_coef=0.65)
registry.ensure("ferry")
registry.remove("ferry")

registry.get("bus")


TransportSpec(name='bus', vmax_tech_kmh=80, accel_dist_m=220, brake_dist_m=160, traffic_coef=0.65)

## Use a registry in graph construction

Pass the registry to public-transport builders through `transport_registry`.


In [6]:
from iduedu import get_public_transport_graph

G_pt_custom = get_public_transport_graph(
    osm_id=OSM_ID,
    transport_types=["bus", "tram"],
    transport_registry=registry,
    clip_by_territory=True,
)

G_pt_custom


Parsing ground PT routes:   0%|          | 0/43 [00:00<?, ?it/s]

UrbanGraph(nodes=935, edges=2056, is_multigraph=True, is_directed=True, edge_direction_column='oneway', crs='EPSG:32636', type='public_transport')